## Basic routine

In [2]:
# Autoload modules
%load_ext autoreload
%autoreload 2
%load_ext line_profiler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import sys
sys.path.append('..')
from fcc import MiyazawaJerniganInteraction, Peptide, ProteinFoldingProblem, PenaltyParameters
from qiskit.circuit.library import RealAmplitudes, EfficientSU2
from qiskit_algorithms.optimizers import COBYLA, SLSQP
from qiskit_algorithms.minimum_eigensolvers import SamplingVQE
from qiskit.primitives import Sampler
import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import Statevector
from sklearn.metrics import r2_score
import pickle

In [2]:
def build_pf(main_seq: str, energy_matrix_file: str = "mj_matrix"):
    """Builds the protein folding problem for the given sequence."""

    mj_interaction = MiyazawaJerniganInteraction(energy_matrix_file)
    # print(mj_interaction.calculate_energy_matrix(main_seq))

    penalty_back = 50
    penalty_redun = 50
    # penalty_olap = 50

    penalty_terms = PenaltyParameters(penalty_back, penalty_redun)

    peptide = Peptide(main_seq)

    protein_folding_problem = ProteinFoldingProblem(peptide, mj_interaction, penalty_terms)

    return protein_folding_problem

In [3]:
main_seq = "LHPGAGK" # Zika
protein_folding_problem = build_pf(main_seq)

qubit_op = protein_folding_problem.qubit_op()
print("Number of qubits: ", qubit_op.num_qubits)
print("Number of terms: ", len(qubit_op))
print(qubit_op)

Number of qubits:  33
Number of terms:  5121
SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZIZZII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIZZZII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZZIZZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZIZIZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZZZIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZIIZIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIZIZIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZZZZZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIZZ', 'IIIIIIIIIIIIIIIIIIIIIIIZZIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIZZIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIZZZZIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIZIIIZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIZIIZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIZIZZZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIZZZZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIZIIIIZIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIZIIIZIIII', 'IIIIIIIIIIIIII

In [4]:

from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit.quantum_info import commutator

def mixer_op_x(n, theta: np.pi):
    # Theta = pi for standard X Pauli
    rx_terms = []
    for i in range(n):
        # Create a Pauli string with X on the i-th qubit and I on the others
        labels = ['I'] * n
        labels[i] = 'X'
        pauli_string = ''.join(labels)
        pauli_op = SparsePauliOp(Pauli(pauli_string), coeffs=[theta/2])
        rx_terms.append(pauli_op)
    mixer_hamiltonian = sum(rx_terms, start=SparsePauliOp.from_list([('I'*n, 0)]))
    return mixer_hamiltonian


def truncate_to_two_local(pauli_op, return_output=True):
    # Extract labels and coefficients
    labels = pauli_op.paulis.to_labels()
    coeffs = pauli_op.coeffs
    # Initialize list to collect the filtered terms
    filtered_labels = []
    filtered_coeffs = []
    # Filter terms to include only those with exactly one or two non-identity operators ('X', 'Y', 'Z')
    for label, coeff in zip(labels, coeffs):
        # Count the number of non-identity Pauli operators
        non_identity_count = sum(1 for char in label if char != 'I')

        # Filter to include only one-weight and two-weight terms
        if non_identity_count == 1 or non_identity_count == 2:
            filtered_labels.append(label)
            filtered_coeffs.append(coeff)
    
    # Construct the new SparsePauliOp if there are any valid terms
    if filtered_labels:
        truncated_terms = SparsePauliOp.from_list(list(zip(filtered_labels, filtered_coeffs)))
    else:
        # Return an empty operator if no terms meet the criteria
        truncated_terms = SparsePauliOp.from_list([('I' * pauli_op.num_qubits, 0)])
    
    # Optionally return the result
    if return_output:
        return truncated_terms
    else:
        # The output can be captured in a variable 'U_cd' for later use without being returned
        U_cd = truncated_terms
        # Optionally print or log the variable for verification or debugging
        # print("U_cd stored:", U_cd)
        pass


def calculate_commutator(hamiltonian, truncate: bool, order: int):
    mixer_op = mixer_op_x(hamiltonian.num_qubits, np.pi)  # Regular Pauli X mixer
    # Set up the Hamiltonians
    H_a = 0.5 * mixer_op + 0.5 * hamiltonian
    partial_H_a = -mixer_op + hamiltonian

    # Start with the first commutator
    current_commutator = commutator(H_a, partial_H_a).simplify()
    
    # Compute nested commutators up to the specified order
    for _ in range(order - 1):
        current_commutator = commutator(H_a, current_commutator).simplify()
    
    if truncate:
        # If truncation is needed, filter to two-local terms with specific Pauli characters
        final_hamiltonian = truncate_to_two_local(current_commutator).simplify()
    else:
        # If no truncation is needed, use the final commutator result
        final_hamiltonian = current_commutator

    return final_hamiltonian

In [5]:
from qiskit.quantum_info import SparsePauliOp
from collections import Counter

def analyze_pauli_terms(pauli_op: SparsePauliOp):
    # Extract labels from the Pauli operator
    labels = pauli_op.paulis.to_labels()
    
    # Initialize a Counter to store occurrences of each Pauli term type
    term_types = Counter()
    
    # Iterate over each label and process
    # Skip the sorting to keep 'XZ' and 'ZX' distinct
    for label in labels:
        # Remove identity matrices, keep the order of non-identity Pauli matrices
        normalized_label = label.replace('I', '')
        if normalized_label == '':
            normalized_label = 'I'  # Include identity as a term if all are 'I'
        term_types[normalized_label] += 1
    
    # Return a dictionary of term types with their counts
    return dict(term_types)


In [6]:
analyze_pauli_terms(qubit_op)

{'I': 1,
 'ZZ': 215,
 'ZZZ': 866,
 'ZZZZ': 1110,
 'ZZZZZZ': 749,
 'ZZZZZZZZ': 3,
 'Z': 29,
 'ZZZZZ': 1588,
 'ZZZZZZZ': 560}

In [7]:
dcd_hamiltonian = calculate_commutator(qubit_op, True, 2)
analyze_pauli_terms(dcd_hamiltonian)

{'Z': 29, 'ZZ': 215, 'YY': 215, 'X': 33, 'XZ': 243, 'ZX': 314}

In [8]:
from qiskit.quantum_info import SparsePauliOp, Pauli

def select_terms(dcd_hamiltonian, term_analysis, num_qubits, selection_mode):
    # Extract labels and coefficients
    labels = dcd_hamiltonian.paulis.to_labels()
    coeffs = dcd_hamiltonian.coeffs
    
    # Initialize variables for lowest count logic if needed
    if selection_mode == "two_local_min":
        lowest_count = float('inf')
        lowest_term_type = None
        for term, count in term_analysis.items():
            if len(term) == 2 and count < lowest_count:
                lowest_count = count
                lowest_term_type = term
    
    # Initialize a list to collect the selected terms with their coefficients
    selected_terms = []
    
    # Filter the terms based on the analysis
    for label, coeff in zip(labels, coeffs):
        # Remove identity and keep original order
        normalized_label = label.replace('I', '')

        # Select weight-1 terms where the count matches the number of qubits
        if normalized_label in term_analysis and len(normalized_label) == 1 and term_analysis[normalized_label] == num_qubits:
            selected_terms.append((label, coeff))
        
        # Select weight-2 terms based on the mode
        if len(normalized_label) == 2:
            if selection_mode == "two_local_min" and normalized_label == lowest_term_type:
                selected_terms.append((label, coeff))
            elif selection_mode == "linear":
                # Check if terms are adjacent non-trivially
                non_trivial_indices = [i for i, x in enumerate(label) if x != 'I']
                if len(non_trivial_indices) == 2 and abs(non_trivial_indices[0] - non_trivial_indices[1]) == 1:
                    selected_terms.append((label, coeff))

    # Return a new SparsePauliOp with the selected terms
    return SparsePauliOp.from_list(selected_terms)


In [9]:
dcd_selection_min = select_terms(dcd_hamiltonian, analyze_pauli_terms(dcd_hamiltonian), qubit_op.num_qubits, "two_local_min")
analyze_pauli_terms(dcd_selection_min)

{'ZZ': 215, 'X': 33}

In [11]:
dcd_selection_linear = select_terms(dcd_hamiltonian, analyze_pauli_terms(dcd_hamiltonian), qubit_op.num_qubits, "linear")
analyze_pauli_terms(dcd_selection_linear)

{'ZZ': 17, 'YY': 17, 'X': 33, 'XZ': 17, 'ZX': 16}

In [12]:
dcd_selection_linear.size

100